# AgentCore 에피소딕 메모리 - 회의록 어시스턴트

이 튜토리얼은 Strands 에이전트와 AgentCore 에피소딕 메모리를 통합하여 회의록 어시스턴트를 구축하는 방법을 보여줍니다. 에이전트는 과거 회의에서 학습하여 내려진 결정, 할당된 액션 아이템, 참석자 선호도를 기억하고, 반복되는 회의에서 컨텍스트 인식 지원을 제공합니다.

튜토리얼 세부사항:
- 튜토리얼 유형: 장기 에피소딕 메모리
- 에이전트 유형: 회의록 어시스턴트
- 에이전트 프레임워크: Strands Agents
- LLM 모델: Anthropic Claude Haiku 4.5
- 구성요소: AgentCore 에피소딕 메모리 및 리플렉션

학습 내용:
- 에피소딕 전략으로 AgentCore 메모리 설정
- 자동 에피소드 캡처를 위한 메모리 훅 생성
- 과거 회의 에피소드 및 리플렉션 검색
- 회의 패턴과 참석자 선호도를 학습하는 에이전트 구축

## 1단계: 종속성 설치 및 설정

In [ ]:
# !pip install -qr requirements.txt

In [ ]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("meeting-notes-assistant")

from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

In [ ]:
# 설정
REGION = "us-west-2"
PARTICIPANT_ID = "participant_001"
SESSION_ID = f"meeting_{datetime.now().strftime('%Y%m%d%H%M%S')}"

## 2단계: 에피소딕 전략으로 메모리 리소스 생성

에피소딕 전략은 자동으로 다음을 수행합니다:
- 대화 내에서 에피소드 완료 시점을 감지
- 구조화된 에피소드 레코드 추출 (상황, 의도, 평가, 근거)
- 에피소드 간 패턴을 식별하는 리플렉션 생성

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=REGION)
memory_name = "MeetingNotesEpisodicMemory"

# 에피소딕 메모리 전략 정의
strategies = [
    {
        StrategyType.EPISODIC.value: {
            "name": "MeetingEpisodes",
            "description": "Captures meeting discussions and generates reflections on meeting patterns",
            "namespaces": ["meetings/actor/{actorId}/episodes"],
            "reflectionConfiguration": {
                "namespaces": ["meetings/actor/{actorId}"]  # 에피소딕 네임스페이스와 동일하거나 접두사여야 합니다
            }
        }
    }
]

# 메모리 리소스 생성
try:
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies,
        description="Episodic memory for meeting notes assistant",
        event_expiry_days=180,  # 단기 메모리 이벤트(STM)의 TTL, 장기 에피소딕 전략용이 아닙니다
    )
    memory_id = memory['id']
    logger.info(f"메모리 생성 완료: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 사용 중: {memory_id}")
    else:
        raise
except Exception as e:
    logger.error(f"오류: {e}")
    raise

In [ ]:
# 에피소딕 전략이 설정되었는지 확인
strategies = client.get_memory_strategies(memory_id)
print(json.dumps(strategies, indent=2, default=str))

## 3단계: 회의 관리 도구 생성

In [ ]:
@tool
def capture_action_item(task: str, owner: str, due_date: str) -> str:
    """Capture an action item from the meeting discussion.
    
    Args:
        task: Description of the task to be completed
        owner: Person responsible for completing the task
        due_date: When the task should be completed
    
    Returns:
        Confirmation of action item capture with details
    """
    action_items = {
        "website": "Website redesign - assigned to Sarah, due next Friday",
        "budget": "Review Q3 budget allocation - assigned to Mike, due this week",
        "presentation": "Prepare stakeholder presentation - assigned to Alex, due Monday",
        "testing": "Complete user testing for new feature - assigned to QA team, due end of sprint",
    }
    
    # 액션 아이템 저장 시뮬레이션
    for keyword, stored_item in action_items.items():
        if keyword in task.lower():
            return f"ACTION ITEM CAPTURED:\n{stored_item}\n\nNote: {task}"
    
    return f"ACTION ITEM CAPTURED:\nTask: {task}\nOwner: {owner}\nDue: {due_date}"


@tool
def identify_decision(decision: str, context: str) -> str:
    """Identify and record a key decision made during the meeting.
    
    Args:
        decision: The decision that was made
        context: Context or reasoning behind the decision
    
    Returns:
        Confirmation of decision recording with summary
    """
    decisions = {
        "budget": "Approved Q3 marketing budget increase of 15%",
        "launch": "Product launch date set for November 15th",
        "vendor": "Selected AWS as cloud infrastructure provider",
        "process": "Adopted agile sprint methodology for project management",
    }
    
    # 결정 기록 시뮬레이션
    for keyword, stored_decision in decisions.items():
        if keyword in decision.lower():
            return f"DECISION RECORDED:\n{stored_decision}\n\nRationale: {context}"
    
    return f"DECISION RECORDED:\n{decision}\n\nContext: {context}"


@tool
def summarize_discussion(topic: str, key_points: str) -> str:
    """Summarize a discussion topic with key points.
    
    Args:
        topic: The discussion topic
        key_points: Main points covered in the discussion
    
    Returns:
        Structured summary of the discussion
    """
    # 토론 요약 시뮬레이션
    return f"""DISCUSSION SUMMARY:

Topic: {topic}

Key Points:
{key_points}

Next Steps: Review in next meeting"""


@tool
def track_followup(previous_item: str, status: str) -> str:
    """Track follow-up status of previous action items or decisions.
    
    Args:
        previous_item: Description of the item to follow up on
        status: Current status (completed, in-progress, blocked, pending)
    
    Returns:
        Follow-up status with details
    """
    # 후속 조치 추적 시뮬레이션
    statuses = {
        "completed": "COMPLETED",
        "in-progress": "IN PROGRESS",
        "blocked": "BLOCKED",
        "pending": "PENDING",
    }
    
    status_emoji = statuses.get(status.lower(), "UNKNOWN")
    
    return f"""{status_emoji}
Item: {previous_item}
Status: {status}
Last Updated: {datetime.now().strftime('%Y-%m-%d')}"""


logger.info("회의 관리 도구 준비 완료")

## 4단계: 에피소딕 메모리 훅 프로바이더 생성

훅 프로바이더는 다음을 수행합니다:
- 쿼리를 처리하기 전에 관련된 과거 회의 에피소드 및 리플렉션을 검색
- 에피소드 추출을 위해 회의 상호작용을 이벤트로 저장
- AgentCore에 의해 에피소드가 자동으로 감지되고 추출됨

In [ ]:
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """메모리 전략에 대한 네임스페이스 매핑을 가져옵니다."""
    strategies = mem_client.get_memory_strategies(memory_id)
    result = {}
    for strategy in strategies:
        reflection_config = strategy.get("reflectionConfiguration", {})
        result[strategy["type"]] = {
            "namespaces": strategy.get("namespaces", []),
            "reflectionNamespaces": reflection_config.get("namespaces", [])
        }
    return result


class EpisodicMemoryHooks(HookProvider):
    """에피소딕 메모리와 리플렉션을 위한 메모리 훅"""
    
    def __init__(self, memory_id: str, client: MemoryClient):
        self.memory_id = memory_id
        self.client = client
        self.namespaces = get_namespaces(self.client, self.memory_id)
    
    def retrieve_episodes_and_reflections(self, event: MessageAddedEvent):
        """처리 전 관련 에피소드 및 리플렉션을 검색합니다."""
        messages = event.agent.messages
        if messages[-1]["role"] != "user" or "toolResult" in messages[-1]["content"][0]:
            return
            
        user_query = messages[-1]["content"][0]["text"]
        actor_id = event.agent.state.get("actor_id")
        
        if not actor_id:
            logger.warning("에이전트 상태에 actor_id가 없습니다")
            return
        
        try:
            all_context = []
            episodic_config = self.namespaces.get("EPISODIC", {})
            
            # 관련 에피소드 검색 ("intent"로 인덱싱됨)
            for namespace_template in episodic_config.get("namespaces", []):
                namespace = namespace_template.format(actorId=actor_id)
                episodes = self.client.retrieve_memories(
                    memory_id=self.memory_id,
                    namespace=namespace,
                    query=user_query,  # 에피소드는 intent로 인덱싱됨
                    top_k=3
                )
                
                for episode in episodes:
                    if isinstance(episode, dict):
                        content = episode.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                all_context.append(f"[PAST EPISODE] {text}")
            
            # 리플렉션 검색 ("use case"로 인덱싱됨)
            for namespace_template in episodic_config.get("reflectionNamespaces", []):
                namespace = namespace_template.format(actorId=actor_id)
                reflections = self.client.retrieve_memories(
                    memory_id=self.memory_id,
                    namespace=namespace,
                    query=user_query,  # 리플렉션은 use case로 인덱싱됨
                    top_k=2
                )
                
                for reflection in reflections:
                    if isinstance(reflection, dict):
                        content = reflection.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                all_context.append(f"[REFLECTION] {text}")
            
            # 컨텍스트를 쿼리에 주입
            if all_context:
                context_text = "\n".join(all_context)
                original_text = messages[-1]["content"][0]["text"]
                messages[-1]["content"][0]["text"] = (
                    f"Past Experience:\n{context_text}\n\nCurrent Query: {original_text}"
                )
                logger.info(f"{len(all_context)}개의 에피소드/리플렉션을 검색했습니다")
                
        except Exception as e:
            logger.error(f"에피소드 검색 실패: {e}")
    
    def save_meeting_interaction(self, event: AfterInvocationEvent):
        """에피소드 추출을 위해 회의 상호작용을 저장합니다."""
        try:
            messages = event.agent.messages
            if len(messages) < 2 or messages[-1]["role"] != "assistant":
                return
            
            # 도구 사용을 포함한 전체 상호작용을 수집
            interaction_messages = []
            for msg in messages:
                role = msg["role"].upper()
                content = msg["content"]
                
                if isinstance(content, list):
                    for item in content:
                        if "text" in item:
                            interaction_messages.append((item["text"], role))
                        elif "toolUse" in item:
                            # 더 나은 에피소드 추출을 위해 도구 사용 포함
                            tool_info = item["toolUse"]
                            tool_text = f"[TOOL: {tool_info.get('name', 'unknown')}]"
                            interaction_messages.append((tool_text, "TOOL"))
                        elif "toolResult" in item:
                            result = item["toolResult"].get("content", [{}])[0].get("text", "")
                            interaction_messages.append((f"[RESULT: {result[:200]}]", "TOOL"))
            
            if interaction_messages:
                actor_id = event.agent.state.get("actor_id")
                session_id = event.agent.state.get("session_id")
                
                if not actor_id or not session_id:
                    logger.warning("actor_id 또는 session_id가 없습니다")
                    return
                
                # 이벤트 저장 - AgentCore가 자동으로 에피소드 완료를 감지합니다
                self.client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=interaction_messages
                )
                logger.info("에피소드 추출을 위해 회의 상호작용을 저장했습니다")
                
        except Exception as e:
            logger.error(f"상호작용 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """에피소딕 메모리 훅을 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.retrieve_episodes_and_reflections)
        registry.add_callback(AfterInvocationEvent, self.save_meeting_interaction)
        logger.info("에피소딕 메모리 훅이 등록되었습니다")

## 5단계: 회의록 에이전트 생성

In [ ]:
episodic_hooks = EpisodicMemoryHooks(memory_id, client)

meeting_agent = Agent(
    hooks=[episodic_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[capture_action_item, identify_decision, summarize_discussion, track_followup],
    state={"actor_id": PARTICIPANT_ID, "session_id": SESSION_ID},
    system_prompt="""You are an expert meeting assistant with memory of past meetings.

Your role:
- Help facilitate productive meetings by tracking decisions and action items
- Use past meeting episodes to provide relevant context and history
- Apply reflections about what works well for different teams and participants
- Remember participant preferences and communication styles

When you see [PAST EPISODE] context, use it to inform your responses.
When you see [REFLECTION] context, apply those learned patterns.

Always:
1. Listen for key decisions and action items
2. Reference relevant past meetings when helpful
3. Track follow-ups on previous action items
4. Summarize discussions clearly and concisely""")

print("에피소딕 메모리가 포함된 회의록 에이전트가 생성되었습니다")

## 6단계: 과거 회의 에피소드 시드 데이터 추가

에피소딕 메모리를 시연하기 위해 이전 회의 세션을 추가합니다.

In [ ]:
# 이전 회의 세션으로 시드 데이터 추가
past_sessions = [
    # 세션 1: 스프린트 계획 회의
    ("Let's plan the Q3 sprint. We need to prioritize features.", "USER"),
    ("I'll help capture the key decisions and action items.", "ASSISTANT"),
    (
        "We should focus on the user authentication feature first. It's blocking other work.",
        "USER",
    ),
    ("[TOOL: identify_decision]", "TOOL"),
    ("[RESULT: 📌 DECISION RECORDED: Prioritize user authentication feature for Q3 sprint]", "TOOL"),
    (
        "Noted. User authentication is the priority. Who will lead this?",
        "ASSISTANT",
    ),
    ("Sarah can handle the authentication work. She has experience with OAuth.", "USER"),
    ("[TOOL: capture_action_item]", "TOOL"),
    ("[RESULT: ✅ ACTION ITEM CAPTURED: Implement user authentication - Sarah - Due: End of sprint]", "TOOL"),
    (
        "Perfect! I've captured that Sarah will implement user authentication by end of sprint.",
        "ASSISTANT",
    ),
    # 세션 2: 예산 검토 회의
    (
        "We need to discuss the Q3 marketing budget. Costs are higher than expected.",
        "USER",
    ),
    ("Let me help track this discussion.", "ASSISTANT"),
    ("[TOOL: summarize_discussion]", "TOOL"),
    ("[RESULT: 📝 DISCUSSION SUMMARY: Q3 marketing budget - costs exceeding projections]", "TOOL"),
    (
        "I propose we increase the budget by 15% to account for the new campaigns.",
        "USER",
    ),
    ("[TOOL: identify_decision]", "TOOL"),
    ("[RESULT: 📌 DECISION RECORDED: Approved Q3 marketing budget increase of 15%]", "TOOL"),
    (
        "Decision captured. Is there a follow-up needed?",
        "ASSISTANT",
    ),
    ("Yes, Mike needs to update the financial forecast by end of week.", "USER"),
    ("[TOOL: capture_action_item]", "TOOL"),
    ("[RESULT: ✅ ACTION ITEM CAPTURED: Update financial forecast - Mike - Due: End of week]", "TOOL"),
]

try:
    client.create_event(
        memory_id=memory_id,
        actor_id=PARTICIPANT_ID,
        session_id="seed_session_001",
        messages=past_sessions
    )
    print("과거 회의 에피소드 시드 데이터가 추가되었습니다")
    print("참고: 에피소드 추출은 백그라운드에서 진행됩니다 (~1분)")
    print("참고: 리플렉션 추출은 에피소드 저장 후 10-15분 소요됩니다")
except Exception as e:
    print(f"이력 시드 데이터 추가 중 오류: {e}")

## 7단계: 회의 시나리오 테스트

이제 에이전트가 과거 에피소드와 리플렉션을 활용해야 합니다.

In [ ]:
# 테스트 1: 이전 결정 후속 조치 - 과거 에피소드를 참조해야 합니다
# 지난주에 논의한 Q3 스프린트 우선순위를 다시 살펴봅시다. 어떤 결정이 내려졌나요?
response1 = meeting_agent("Let's revisit the Q3 sprint priorities we discussed last week. What was decided?")
print(f"에이전트: {response1}")

In [ ]:
# 테스트 2: 액션 아이템 확인 - 과거 액션 아이템을 검색해야 합니다
# 사용자 인증 기능을 담당하는 사람을 배정했나요?
response2 = meeting_agent("Did we assign someone to handle the user authentication feature?")
print(f"에이전트: {response2}")

In [ ]:
# 테스트 3: 예산 후속 조치 - 과거 예산 논의를 참조해야 합니다
# Q3 마케팅 예산 논의의 결과는 무엇이었나요?
response3 = meeting_agent("What was the outcome of the Q3 marketing budget discussion?")
print(f"에이전트: {response3}")

In [ ]:
# 테스트 4: 여러 액션이 포함된 새 회의
# 제품 출시 계획 회의를 진행합니다. 주요 사항:
# - 출시일: 11월 15일
# - 마케팅 팀은 2주 준비 시간 필요
# - Sarah가 벤더 조율 담당
# - Mike는 다음 금요일까지 가격 확정 필요
# 결정과 액션 아이템을 캡처해 주세요.
response4 = meeting_agent("""We're having a product launch planning meeting. Key points:
- Launch date: November 15th
- Marketing team needs 2 weeks prep time
- Sarah will coordinate with vendors
- Mike needs to finalize pricing by next Friday

Can you capture the decisions and action items?""")
print(f"에이전트: {response4}")

In [ ]:
# 테스트 5: 패턴 인식 - 에이전트가 참석자 선호도를 기억해야 합니다
# Sarah가 새 기능의 기술 아키텍처를 논의하고 싶어합니다. 어떤 형식이 가장 적합한가요?
response5 = meeting_agent("Sarah wants to discuss technical architecture for the new feature. What format works best?")
print(f"에이전트: {response5}")

In [ ]:
# 테스트 6: 완전히 새로운 주제 - 과거 컨텍스트 없음
# 회사의 지속가능성 이니셔티브에 대해 처음으로 논의해야 합니다. 아이디어를 브레인스토밍해 봅시다.
response6 = meeting_agent("We need to discuss the company's sustainability initiative for the first time. Let's brainstorm ideas.")
print(f"에이전트: {response6}")

## 8단계: 에피소드 저장 확인

In [ ]:
print("\n에피소딕 메모리 요약:")
print("=" * 50)

episodic_config = get_namespaces(client, memory_id).get("EPISODIC", {})

# 에피소드 확인
for namespace_template in episodic_config.get("namespaces", []):
    namespace = namespace_template.format(actorId=PARTICIPANT_ID)
    
    try:
        episodes = client.retrieve_memories(
            memory_id=memory_id,
            namespace=namespace,
            query="meeting decisions action items",
            top_k=5
        )
        
        print(f"\n에피소드 ({len(episodes)}개 발견):")
        for i, episode in enumerate(episodes, 1):
            if isinstance(episode, dict):
                content = episode.get('content', {})
                if isinstance(content, dict):
                    text = content.get('text', '')[:200] + "..."
                    print(f"  {i}. {text}")
                
    except Exception as e:
        print(f"에피소드 검색 오류: {e}")

# 리플렉션 확인
for namespace_template in episodic_config.get("reflectionNamespaces", []):
    namespace = namespace_template.format(actorId=PARTICIPANT_ID)
    
    try:
        reflections = client.retrieve_memories(
            memory_id=memory_id,
            namespace=namespace,
            query="meeting patterns effectiveness",
            top_k=3
        )
        
        print(f"\n리플렉션 ({len(reflections)}개 발견):")
        for i, reflection in enumerate(reflections, 1):
            if isinstance(reflection, dict):
                content = reflection.get('content', {})
                if isinstance(content, dict):
                    text = content.get('text', '')[:200] + "..."
                    print(f"  {i}. {text}")
                
    except Exception as e:
        print(f"리플렉션 검색 오류: {e}")

print("\n" + "=" * 50)

## 주요 시사점

1. **에피소딕 메모리는 상호작용 시퀀스를 캡처합니다**, 단순한 사실이 아닙니다
2. **리플렉션은 여러 에피소드를 분석하여 자동으로 생성됩니다**
3. **이벤트에 도구 결과를 포함하면** 더 풍부한 에피소드 추출이 가능합니다
4. 에피소드는 **intent로 쿼리**하고, 리플렉션은 **use case로 쿼리**합니다
5. **에피소드 추출은 비동기적**입니다 (대화 종료 후 ~1분)

## 정리

In [ ]:
# 메모리 리소스를 삭제하려면 주석을 해제하세요
# try:
#     client.delete_memory_and_wait(memory_id=memory_id)
#     print(f"메모리 리소스 삭제 완료: {memory_id}")
# except Exception as e:
#     print(f"메모리 삭제 오류: {e}")